# Legal Contract QA Pipeline

**Clean → Ollama (`qwen3.5:0.8b`) → span alignment → metrics / submission**

Setup: `uv sync`, `ollama pull qwen3.5:0.8b`, kernel **qa-project-2**. Keep Ollama running for inference.

In [ ]:
from run_qa_pipeline import (
    DATA_DIR,
    OUTPUT_DIR,
    OLLAMA_MODEL,
    build_run_config,
    clean_text,
    create_run_dir,
    exact_match,
    load_data,
    parse_question,
    predict_row,
    resolve_ollama_model,
    retrieve_excerpt,
    run_test,
    run_validation,
    save_cleaning_samples,
    save_run_artifacts,
)

MODEL = resolve_ollama_model(OLLAMA_MODEL)
RUN_DIR = create_run_dir(MODEL, mode="notebook")
print("Model:", MODEL)
print("Run dir:", RUN_DIR)

## 1. Load data & EDA

In [ ]:
train, test = load_data()
print(f"Train: {len(train)}, Test: {len(test)}")
print(train.columns.tolist())

train["context_len"] = train["context"].str.len()
print(train["context_len"].describe())

print("QuickLinks:", train["context"].str.contains("QuickLinks", case=False, na=False).sum())
print("Signatures /s/:", train["context"].str.contains(r"/s/", regex=True, na=False).sum())

train[["id", "question", "answers"]].head(3)

## 2. Text cleaning — before / after examples

In [ ]:
save_cleaning_samples(train, RUN_DIR, n=3)

for i in range(3):
    raw = train.iloc[i]["context"]
    cleaned = clean_text(raw)
    clause, detail = parse_question(train.iloc[i]["question"])
    print(f"--- id={train.iloc[i]['id']} | clause={clause!r} ---")
    print("BEFORE:", repr(raw[:200]))
    print("AFTER: ", repr(cleaned[:200]))
    print()

## 3. Retrieval + single-row Ollama test

In [ ]:
row = train.iloc[0]
clause, detail = parse_question(row["question"])
excerpt = retrieve_excerpt(clean_text(row["context"]), clause, detail)
print("Clause:", clause)
print("Detail:", detail)
print("Excerpt length:", len(excerpt))
print(excerpt[:500], "...")

aligned, raw, exact_sub = predict_row(row["context"], row["question"])
print("\nGold:  ", row["answers"])
print("Raw:   ", raw[:300])
print("Aligned:", aligned[:300])
print("Exact match:", exact_match(aligned, row["answers"]))

## 4. Validation (80/20) — metrics & predictions

In [ ]:
run_config = build_run_config(
    model=MODEL,
    mode="notebook",
    use_cache=True,
    limit=None,
    refresh_cache=False,
    run_dir=RUN_DIR,
)
save_run_artifacts(RUN_DIR, run_config, train)

pred_df, metrics = run_validation(train, model=MODEL, use_cache=True)
pred_df.to_csv(RUN_DIR / "val_predictions.csv", index=False)
save_run_artifacts(RUN_DIR, run_config, train, metrics=metrics)

print(f"Validation exact match: {metrics['val_exact_match']:.4f} ({metrics['n_val']} examples)")
print(f"Saved to {RUN_DIR}")
pred_df[pred_df["exact_match"]].head(3)

In [ ]:
pred_df[~pred_df["exact_match"]].head(5)

## 5. Test set → Kaggle submission

In [ ]:
submission = run_test(test, model=MODEL, use_cache=True)
submission.to_csv(RUN_DIR / "submission.csv", index=False)
print(f"Wrote {RUN_DIR / 'submission.csv'} ({len(submission)} rows)")
submission.head()